# Voyage pilot: voyage-4-nano as a third teacher

**Why.** The student (karma v3, 21,600 steps, TR-MTEB 68.11) is at its teachers' level on retrieval. The remaining gap to Qwen3-8B (~1.2 Mean(task) points) sits in four tasks where both teachers are weak:

| task | student | BGE-M3 | 8K-embed | Qwen3-8B | voyage-4-nano |
|---|---|---|---|---|---|
| SciFact | 78.5 | 78.8 | 75.9 | 87.9 | 86.4 |
| ArguAna | 48.4 | 50.5 | 49.4 | 59.9 | |
| FiQA | 45.2 | 45.0 | 41.7 | 52.1 | |
| CQADupstack | 60.5 | 58.4 | 53.2 | 64.0 | |

voyage-4-nano (346M, Qwen3-based, 2048-d) adds +0.81 to the BGE + 8K per-task oracle, mostly retrieval. Qwen3-8B added more (+1.57) and hurt in training, so this is a pilot.

**Steps** (A100; re-running resumes and skips finished parts):
1. Encode voyage on the 200K mix (plain, no instruction). The BGE and 8K caches exist.
2. Geometry screen (CPU, minutes): voyage next to BGE, 8K and Qwen3-8B on the first 10,000 texts. Qwen failed with 787 PCs for 90% of the variance (BGE 319, 8K 211).
3. Three runs (200K mix, lr 1e-4, 5400 steps, seed 42):
   - `voyage_single_s5400`: voyage alone. The other single-teacher students at lr 1e-4 are BGE 66.07 and 8K 65.67; Qwen alone gave 63.09 (lr 5e-5, against 65.48).
   - `three_voyage_s5400`: BGE + plain 8K + voyage, equal weights.
   - `three_voyage05_s5400`: the same with voyage weight 0.5.

**Reference:** the plain pair `scale_two_lr1e4_mix200k_s5400`, TR-MTEB 67.16 (seed 43: 67.25). Only differences above 0.3 are read; look at SciFact / ArguAna / FiQA / CQADupstack and at Pair and STS (where Qwen hurt). If a three-teacher run clears 67.46, the next step is the main line (karma v3, 21,600 steps; voyage then needs the FineWeb half encoded).


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)


In [ ]:
# --- 1. Encode voyage-4-nano on the 200K mix (resumable at shard level) ---
!python -m turkish_distill.encode_teachers --config configs/three_bge_8k_voyage.yaml --status
!python -m turkish_distill.encode_teachers --config configs/three_bge_8k_voyage.yaml --only voyage_4_nano


In [ ]:
# --- 2. Geometry screen: voyage next to BGE-M3, 8K-embed and Qwen3-8B (first 10,000 texts of the 200K mix) ---
MIX200K = '/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl'
os.makedirs(f'{DRIVE_ROOT}/diag', exist_ok=True)
!python -m turkish_distill.teacher_compare --config configs/three_bge_8k_voyage.yaml --n 10000 --out {DRIVE_ROOT}/diag/geometry_voyage_mix200k.json
print('Qwen3-8B for comparison (its 200K cache exists):')
!python -m turkish_distill.teacher_compare --config configs/stage1_three_bge_m3_modernbert_tr_embed_qwen3_8b.yaml --override data.corpus_path={MIX200K} --n 10000 --out {DRIVE_ROOT}/diag/geometry_qwen_mix200k.json


In [ ]:
# --- 3. Training runs ---
# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
# Both configs already hold the recipe (200K mix, lr 1e-4, 5400 steps).
THREE = 'configs/three_bge_8k_voyage.yaml'   # teachers.0 = bge_m3, .1 = modernbert_tr_embed, .2 = voyage_4_nano
BASE = []
RUNS = [
    ('voyage_single_s5400', 'configs/single_voyage.yaml', []),
    ('three_voyage_s5400', THREE, []),
    ('three_voyage05_s5400', THREE, ['teachers.2.weight=0.5']),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()